In [1]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from notebookutils import mssparkutils

StatementMeta(, 2f1d8b54-720b-40cb-a70a-abcc3de1ef60, 3, Finished, Available, Finished, False)

LOAD COMMON FUNCTION FOR LOG 

In [ ]:
%run /nb_common_utils

In [ ]:
# load_type = p_load_type.upper()   # FULL | INCREMENTAL
# p_config_load_type = "INCREMENTAL"
# p_default_load_type = "FULL"
# p_file_name = "products.csv"

In [ ]:
ctx = log_start(
    spark,
    run_id    = run_id,
    file_name = "products.csv",
    entity    = "product",
    load_type = load_type
)

In [2]:


load_type = (
    p_config_load_type.strip()
    if p_config_load_type and p_config_load_type.strip() != ""
    else p_default_load_type.strip()
)

file_name = p_file_name

print(f"Current load type: {load_type}")


df_raw_prod = spark.read.option("header", True).option("inferSchema", True) \
    .csv(f'Files/raw_uploads/{file_name}')

StatementMeta(, 2f1d8b54-720b-40cb-a70a-abcc3de1ef60, 4, Finished, Available, Finished, False)

Current load type: INCREMENTAL


In [10]:
# clean / transform
df_clean_prod = (
    df_raw_prod
        .filter(F.col("product_id").isNotNull())
        .dropDuplicates(["product_id"])
        .withColumn("unit_price", F.col("unit_price").cast("double"))
        .filter(F.col("unit_price") > 0)
        .withColumn("last_modified", F.current_timestamp())
        .withColumn("updated_at", F.current_timestamp())
)

# cache reuse 
df_clean_prod.cache()

target_table = "silver.clean_products"
stage_table  = "stage.stg_new_products"
compare_cols = ["product_name", "category", "unit_price"]

def build_stage_from_batch(df_batch, target_table, compare_cols):

    if not spark.catalog.tableExists(target_table):
        return df_batch.withColumn("change_type", F.lit("INSERT"))
    
    batch_ids = df_batch.select("product_id")

    df_existings = (
        spark.table(target_table).join(
            batch_ids, on="product_id", how="inner"
        )
    )

    df_new = (
        df_batch.alias("s")
        .join(df_existings.select("product_id").alias("t"),
              on="product_id", how="left_anti")
        .withColumn("change_type", F.lit("INSERT"))
    )

    # CHANGED: same ID but columns change
    changed_cond = F.lit(False)
    for col in compare_cols:
        changed_cond = changed_cond | (F.col(f"s.{col}") != F.col(f"t.{col}"))
    
    df_changed = (
        df_batch.alias("s")
        .join(df_existings.alias("t"), on="product_id", how="inner")
        .filter(changed_cond)
        .select(
            F.col("s.product_id"),
            F.col("s.product_name"),
            F.col("s.category"),
            F.col("s.unit_price"),
            F.col("s.last_modified"),
            F.col("s.updated_at"),
        )
        .withColumn("change_type", F.lit("UPDATE"))
    )
    
    return df_new.unionByName(df_changed)



StatementMeta(, 2f1d8b54-720b-40cb-a70a-abcc3de1ef60, 12, Finished, Available, Finished, False)

Main

In [11]:
if load_type == "FULL":
    print("Running FULL LOAD...")

    # Previous Stage
    df_stage = df_clean_prod.withColumn("change_type", F.lit("UPSERT"))
    (
        df_stage.write
            .mode("overwrite")
            .format("delta")
            .saveAsTable(stage_table)
    )

    print(f"Stage written: {df_stage.count()} rows")

    (
        df_clean_prod.write
            .mode("overwrite")
            .format("delta")
            .saveAsTable(target_table)
    )

    print("FULL LOAD completed")

elif load_type == "INCREMENTAL":
    
    print("Running INCREMENTAL LOAD...")

    df_stage = build_stage_from_batch(df_clean_prod, target_table, compare_cols)

    stage_count = df_stage.count()

    print(f"Changes detected: {stage_count} rows")

    # Write into stage
    if stage_count > 0:
        (
            df_stage.write
                .mode("overwrite")
                .format("delta")
                .saveAsTable(stage_table)
        )
        print("Stage OVERWRITE completed")
    else:
        print("No changes — stage not updated")


    # merge clean 
    if spark.catalog.tableExists(target_table):

        delta_target = DeltaTable.forName(spark, target_table)
        (
            delta_target.alias("t")
            .merge(df_clean_prod.alias("s"), "t.product_id = s.product_id")
            .whenMatchedUpdate(set={
                "product_name": "s.product_name",
                "category":     "s.category",
                "unit_price":   "s.unit_price",
                "last_modified": "s.last_modified",
                "updated_at":   "s.updated_at",
            })
            .whenNotMatchedInsert(values={
                "product_id":   "s.product_id",
                "product_name": "s.product_name",
                "category":     "s.category",
                "unit_price":   "s.unit_price",
                "last_modified": "s.last_modified",
                "updated_at":   "s.updated_at",
            })
            .execute()
        )
        
        print("INCREMENTAL MERGE into clean completed")


    else:
        (
            df_clean_prod.write
                .mode("overwrite")
                .format("delta")
                .saveAsTable(target_table)
        )
        print("Initial load into clean completed")
else:
    raise Exception(f"Unsupported load type: {load_type}")

#clean cache
df_clean_prod.unpersist()


StatementMeta(, 2f1d8b54-720b-40cb-a70a-abcc3de1ef60, 13, Finished, Available, Finished, False)

Running INCREMENTAL LOAD...
Changes detected: 30 rows
Stage OVERWRITE completed
Initial load into clean completed
